# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id`.

We'll list all available record sets (`RecordSet`), their fields (`Field` and related columns), and show their `@id` values.


In [ ]:
# Fetch all record sets with their @id
record_sets = dataset.record_sets

print("There are %d record set(s) in the dataset.\n" % len(record_sets))
for rset in record_sets:
    print(f"RecordSet @id: {rset.id}\n  name: {rset.name or '[no name]'}\n  description: {rset.description or '[no description]'}")
    print("  Fields and columns:")
    for field in rset.fields:
        col_ids = [c.id for c in getattr(field, 'columns', [])]
        print(f"    Field @id: {field.id} | label: {getattr(field, 'name', None)} | data_type: {getattr(field, 'data_type', None)} | columns: {col_ids}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we'll choose each available record set by its `@id` and extract all records into a pandas DataFrame.

In [ ]:
# Gather all record set IDs
all_record_set_ids = [rset.id for rset in dataset.record_sets]

dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  → Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  → No records found for this record set.")

#### Preview the first few records of the main record set

For demonstration, let’s select the first record set found (by `@id`):

In [ ]:
# Select the first available dataframe for demonstration
if len(dataframes):
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Previewing main record set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes were loaded; check the record sets.")

## 4. Exploratory Data Analysis (EDA)
Let us apply data processing steps—filtering, normalizing, and grouping.

Select a numeric field by its `@id`, filter by a threshold, normalize, and group by a categorical field.

> If you have multiple record sets, adjust the variables below to target your main analytical table and a suitable numeric and group/categorical field.

In [ ]:
# Specify the main record set and fields by their `@id`

# Replace these values with the appropriate @ids as printed above
record_set_id = main_record_set_id  # from the above cell

# Pick a likely numeric field and a group field by inspecting the columns
df = dataframes[record_set_id]
print("Available columns:", df.columns.tolist())

# Example: Use 'cr:protein_abundance' as numeric; 'cr:sample_id' as group (replace according to actual column names)
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if (numeric_field_id is None) and (df[col].dtype.kind in 'fi'):
        numeric_field_id = col
    if (group_field_id is None) and ('sample' in col.lower() or 'group' in col.lower()):
        group_field_id = col
if numeric_field_id is None:
    numeric_field_id = df.columns[0]
if group_field_id is None:
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

# Clean up numeric field
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records
threshold = df[numeric_field_id].median() if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize
if filtered_df[numeric_field_id].std() != 0 and filtered_df[numeric_field_id].notnull().any():
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by group field if available
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (showing group mean of numeric field):")
    display(grouped_df.head())

## 5. Visualization
We can visualize the distribution of the numeric field and show group-wise statistics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group if available
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
With `mlcroissant`, we loaded the Mass Spectrometry FAIR² dataset, previewed its structure, inspected available record sets, extracted data using Croissant `@id`s, and performed a simple EDA and visualization.

This process can be extended to domain-specific questions such as identifying outliers in protein abundance, correlating modifications with samples, or exporting cleaned data for advanced modeling. Refer to the Croissant schema for in-depth exploration of fields using their unique `@id` identifiers.